# 🧠 Phase 3 — Modélisation : Autoencodeur 3D sur Nuages de Points
**Projet : Détection d'Anomalies dans des Objets Industriels 3D — Catégorie : bagel**

---
**⚠️ Avant de démarrer : Activer le GPU**
```
Menu : Exécution → Modifier le type d'exécution → GPU (T4)
```

**Architecture choisie : Autoencodeur basé sur PointNet**
- **Encoder** : couches MLP partagées + MaxPooling global → vecteur latent z
- **Decoder** : MLP → reconstruction du nuage de points
- **Loss** : Chamfer Distance (mesure l'écart entre nuage réel et reconstruit)
- **Principe de détection** : anomalie = erreur de reconstruction élevée

**Résultats Phase 2 :**
- Tenseur : `(batch=16, N=2048, 3)` float32
- Train : 244 bagels normaux | Val : 22 | Test : 110

## 1. Setup et vérification GPU

In [ ]:
!pip install open3d tifffile torch torchvision scikit-learn matplotlib -q

import os, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import tifffile

# Vérification GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Pas de GPU détecté — activez-le dans Exécution → Modifier le type d\'exécution')

# Reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed(SEED)

# ─── Hyperparamètres ──────────────────────────────────────────────────────
N_POINTS    = 2048
BATCH_SIZE  = 16
LATENT_DIM  = 128    # dimension du vecteur latent
N_EPOCHS    = 100    # nombre d'époques d'entraînement
LR          = 1e-3   # learning rate Adam
CATEGORY    = 'bagel'
EXTRACT_DIR = '/content/mvtec3d'
CAT_PATH    = Path(EXTRACT_DIR) / CATEGORY

print(f'\n✅ Hyperparamètres :')
print(f'   N_POINTS   = {N_POINTS}')
print(f'   LATENT_DIM = {LATENT_DIM}')
print(f'   N_EPOCHS   = {N_EPOCHS}')
print(f'   LR         = {LR}')

## 2. Reprise du pipeline Phase 2 (DataLoaders)

In [ ]:
# ── Fonctions de prétraitement (reprises de la Phase 2) ───────────────────
def load_xyz_tiff(path):
    xyz_map = tifffile.imread(str(path))
    mask    = np.any(xyz_map != 0, axis=-1)
    return xyz_map[mask].astype(np.float32)

def normalize_pointcloud(points):
    centroid  = points.mean(axis=0)
    points    = points - centroid
    max_dist  = np.max(np.linalg.norm(points, axis=1))
    return (points / max_dist).astype(np.float32), centroid, max_dist

def random_sample(points, n_points=2048):
    n = len(points)
    idx = np.random.choice(n, n_points, replace=(n < n_points))
    return points[idx]

def augment_pointcloud(points):
    theta = np.random.uniform(0, 2 * np.pi)
    c, s  = np.cos(theta), np.sin(theta)
    R     = np.array([[c, -s, 0],[s, c, 0],[0, 0, 1]], dtype=np.float32)
    points = (points @ R.T)
    points = points + np.random.normal(0, 0.005, points.shape).astype(np.float32)
    return (points * np.random.uniform(0.9, 1.1)).astype(np.float32)

def collect_samples(cat_path, split):
    samples = []
    split_path = cat_path / split
    if not split_path.exists():
        return samples
    for defect_dir in sorted(split_path.iterdir()):
        if not defect_dir.is_dir():
            continue
        is_good  = (defect_dir.name == 'good')
        xyz_dir  = defect_dir / 'xyz'
        gt_dir   = defect_dir / 'gt'
        if not xyz_dir.exists():
            continue
        for xyz_file in sorted(xyz_dir.glob('*.tiff')):
            gt_file = None
            if not is_good and gt_dir.exists():
                cands = list(gt_dir.glob(f'{xyz_file.stem}*.png'))
                if cands:
                    gt_file = cands[0]
            samples.append({'xyz': xyz_file, 'label': 0 if is_good else 1,
                             'defect_type': defect_dir.name, 'gt': gt_file})
    return samples

class BagelDataset(Dataset):
    def __init__(self, samples, n_points=2048, training=False, augment=False):
        self.samples  = samples
        self.n_points = n_points
        self.training = training
        self.augment  = augment

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s      = self.samples[idx]
        pts, _, _ = normalize_pointcloud(load_xyz_tiff(s['xyz']))
        pts    = random_sample(pts, self.n_points)
        if self.augment and self.training:
            pts = augment_pointcloud(pts)
        return (torch.from_numpy(pts).float(),
                torch.tensor(s['label']).long(),
                s['defect_type'])

# Construction des DataLoaders
train_samples = collect_samples(CAT_PATH, 'train')
val_samples   = collect_samples(CAT_PATH, 'validation')
test_samples  = collect_samples(CAT_PATH, 'test')

train_loader = DataLoader(BagelDataset(train_samples, N_POINTS, training=True,  augment=True),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(BagelDataset(val_samples,   N_POINTS, training=False, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(BagelDataset(test_samples,  N_POINTS, training=False, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'✅ DataLoaders reconstruits : train={len(train_loader)} batchs | val={len(val_loader)} | test={len(test_loader)}')

## 3. Architecture — Autoencodeur PointNet

```
Input (B, N, 3)
     │
  ENCODER
  MLP partagé : 3 → 64 → 128 → 256
  MaxPool global → vecteur latent (B, LATENT_DIM)
     │
  DECODER
  MLP : LATENT_DIM → 256 → 512 → N×3
  Reshape → (B, N, 3)
     │
  LOSS : Chamfer Distance
  Score anomalie = CD(input, reconstruction)
```

In [ ]:
class PointNetEncoder(nn.Module):
    """
    Encoder PointNet : applique des MLP partagés point par point,
    puis agrège avec un MaxPool global → vecteur latent invariant
    à l'ordre et au nombre de points.
    """
    def __init__(self, latent_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            # Couche 1 : 3 → 64
            nn.Conv1d(3,   64,  1), nn.BatchNorm1d(64),  nn.ReLU(),
            # Couche 2 : 64 → 128
            nn.Conv1d(64,  128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            # Couche 3 : 128 → 256
            nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        # Projection vers l'espace latent
        self.fc = nn.Sequential(
            nn.Linear(256, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        # x : (B, N, 3) → transpose pour Conv1d : (B, 3, N)
        x = x.transpose(2, 1)        # (B, 3, N)
        x = self.mlp(x)              # (B, 256, N)
        x = x.max(dim=2)[0]          # MaxPool global : (B, 256)
        z = self.fc(x)               # vecteur latent : (B, latent_dim)
        return z


class PointNetDecoder(nn.Module):
    """
    Decoder : reconstruit le nuage de points depuis le vecteur latent.
    Approche Fold-based : répète le vecteur latent N fois et applique un MLP.
    """
    def __init__(self, latent_dim=128, n_points=2048):
        super().__init__()
        self.n_points = n_points
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, 256),  nn.ReLU(),
            nn.Linear(256,        512),  nn.ReLU(),
            nn.Linear(512,        1024), nn.ReLU(),
            nn.Linear(1024, n_points * 3)   # sortie : N×3 coordonnées
        )

    def forward(self, z):
        # z : (B, latent_dim)
        out = self.mlp(z)                         # (B, N*3)
        return out.view(-1, self.n_points, 3)      # (B, N, 3)


class PointNetAutoEncoder(nn.Module):
    """
    Autoencodeur complet : Encoder + Decoder.
    En inférence, le score d'anomalie est la Chamfer Distance
    entre l'entrée et la reconstruction.
    """
    def __init__(self, latent_dim=128, n_points=2048):
        super().__init__()
        self.encoder = PointNetEncoder(latent_dim)
        self.decoder = PointNetDecoder(latent_dim, n_points)

    def forward(self, x):
        z    = self.encoder(x)   # (B, latent_dim)
        recon = self.decoder(z)  # (B, N, 3)
        return recon, z


# Instanciation et résumé
model = PointNetAutoEncoder(latent_dim=LATENT_DIM, n_points=N_POINTS).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Modèle instancié sur {DEVICE}')
print(f'   Paramètres entraînables : {n_params:,}')
print(f'   Latent dim              : {LATENT_DIM}')
print(f'   Points en entrée        : {N_POINTS}')

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(2, N_POINTS, 3).to(DEVICE)
    recon, z = model(dummy)
    print(f'\n   Input shape  : {dummy.shape}')
    print(f'   Latent shape : {z.shape}    (vecteur latent)')
    print(f'   Output shape : {recon.shape}  (reconstruction)')

## 4. Fonction de loss — Chamfer Distance

In [ ]:
def chamfer_distance(pc1, pc2):
    """
    Chamfer Distance entre deux nuages de points.

    Pour chaque point de pc1, cherche le point le plus proche dans pc2
    et vice versa. La CD est la somme des deux distances moyennes.

    Args:
        pc1, pc2 : tenseurs (B, N, 3)
    Returns:
        cd_loss : scalaire (distance moyenne sur le batch)
        cd_per_sample : (B,) distance par pièce (pour le score d'anomalie)
    """
    B, N, _ = pc1.shape

    # Matrice des distances au carré entre tous les paires de points
    # (B, N, 1, 3) - (B, 1, M, 3) → (B, N, M)
    diff = pc1.unsqueeze(2) - pc2.unsqueeze(1)   # (B, N, M, 3)
    dist = (diff ** 2).sum(dim=-1)               # (B, N, M)

    # Distance minimale : chaque point trouve son plus proche voisin
    dist1 = dist.min(dim=2)[0]    # (B, N) : pour chaque pt de pc1 → min dans pc2
    dist2 = dist.min(dim=1)[0]    # (B, M) : pour chaque pt de pc2 → min dans pc1

    # CD par échantillon (pour le score d'anomalie)
    cd_per_sample = dist1.mean(dim=1) + dist2.mean(dim=1)   # (B,)

    # CD moyenne sur le batch (pour la loss d'entraînement)
    cd_loss = cd_per_sample.mean()

    return cd_loss, cd_per_sample


# Test de la Chamfer Distance
with torch.no_grad():
    pc_a = torch.randn(4, 512, 3).to(DEVICE)
    pc_b = pc_a + 0.01   # légèrement différent
    cd, cd_per = chamfer_distance(pc_a, pc_b)
    print(f'✅ Chamfer Distance — test :')
    print(f'   CD (nuages proches)    : {cd.item():.6f}  (doit être faible)')

    pc_c = torch.randn(4, 512, 3).to(DEVICE)   # complètement différent
    cd2, _ = chamfer_distance(pc_a, pc_c)
    print(f'   CD (nuages aléatoires) : {cd2.item():.6f}  (doit être élevé)')
    print(f'   → La CD distingue bien normal vs anormal ✅')

## 5. Entraînement

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-5)

# Historique pour les courbes
history = {'train_loss': [], 'val_loss': [], 'epoch_time': []}

SAVE_PATH = '/content/drive/MyDrive/Detection_Anomalie_3D/models'
os.makedirs(SAVE_PATH, exist_ok=True)
best_val_loss = float('inf')

print(f'🚀 Début entraînement — {N_EPOCHS} époques sur {DEVICE}')
print(f'   Optimizer : Adam (lr={LR})')
print(f'   Scheduler : CosineAnnealing')
print('-' * 55)

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ──────────────────────────────────────────────────────────
    model.train()
    train_losses = []
    for pts, labels, _ in train_loader:
        pts = pts.to(DEVICE)               # (B, N, 3)
        optimizer.zero_grad()
        recon, _ = model(pts)              # (B, N, 3)
        loss, _  = chamfer_distance(pts, recon)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # ── VALIDATION ─────────────────────────────────────────────────────
    model.eval()
    val_losses = []
    with torch.no_grad():
        for pts, labels, _ in val_loader:
            pts = pts.to(DEVICE)
            recon, _ = model(pts)
            loss, _  = chamfer_distance(pts, recon)
            val_losses.append(loss.item())

    scheduler.step()

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    epoch_time = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['epoch_time'].append(epoch_time)

    # Sauvegarde du meilleur modèle
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),
                   os.path.join(SAVE_PATH, f'{CATEGORY}_best_model.pth'))
        saved = '💾 saved'
    else:
        saved = ''

    # Affichage toutes les 10 époques
    if epoch % 10 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{N_EPOCHS} | '
              f'Train: {train_loss:.6f} | '
              f'Val: {val_loss:.6f} | '
              f'{epoch_time:.1f}s {saved}')

print(f'\n✅ Entraînement terminé !')
print(f'   Meilleure val loss : {best_val_loss:.6f}')
print(f'   Temps moyen/époque : {np.mean(history["epoch_time"]):.1f}s')

## 6. Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Courbe de loss
epochs = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs, history['train_loss'], label='Train loss', color='#2196F3', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val loss',   color='#FF5722', linewidth=2)
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Chamfer Distance')
axes[0].set_title('Courbes de loss — Chamfer Distance')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Zoom sur la convergence (après époque 10)
if len(history['train_loss']) > 10:
    axes[1].plot(list(epochs)[10:], history['train_loss'][10:],
                 label='Train', color='#2196F3', linewidth=2)
    axes[1].plot(list(epochs)[10:], history['val_loss'][10:],
                 label='Val',   color='#FF5722', linewidth=2)
    axes[1].set_title('Convergence (époque 10 → fin)')
    axes[1].set_xlabel('Époque')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.suptitle(f'Autoencodeur PointNet — bagel — {N_EPOCHS} époques', fontsize=12)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Courbes sauvegardées : /content/training_curves.png')

## 7. Chargement du meilleur modèle et scores d'anomalie

In [ ]:
# Charger le meilleur modèle sauvegardé
model.load_state_dict(
    torch.load(os.path.join(SAVE_PATH, f'{CATEGORY}_best_model.pth'),
               map_location=DEVICE)
)
model.eval()
print('✅ Meilleur modèle chargé')


def compute_anomaly_scores(loader, model, device):
    """
    Calcule le score d'anomalie pour chaque pièce.
    Score = Chamfer Distance entre entrée et reconstruction.
    Une pièce défectueuse → reconstruction pauvre → score élevé.
    """
    all_scores  = []
    all_labels  = []
    all_defects = []

    with torch.no_grad():
        for pts, labels, defects in loader:
            pts   = pts.to(device)
            recon, _ = model(pts)
            _, cd_per_sample = chamfer_distance(pts, recon)

            all_scores.extend(cd_per_sample.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
            all_defects.extend(list(defects))

    return np.array(all_scores), np.array(all_labels), all_defects


# Scores sur validation (pour calibrer le seuil)
val_scores, val_labels, val_defects = compute_anomaly_scores(val_loader, model, DEVICE)

# Scores sur test (évaluation finale)
test_scores, test_labels, test_defects = compute_anomaly_scores(test_loader, model, DEVICE)

print(f'\n✅ Scores calculés :')
print(f'   Validation : {len(val_scores)} scores')
print(f'   Test       : {len(test_scores)} scores')
print(f'\n   Scores sur test :')
for dtype in ['good', 'crack', 'hole', 'contamination', 'combined']:
    mask = [d == dtype for d in test_defects]
    s = np.array(test_scores)[mask]
    if len(s) > 0:
        print(f'     {dtype:15s} → mean={s.mean():.6f}  std={s.std():.6f}')

## 8. Calibration du seuil de détection

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score

# ── Calibration sur la validation ─────────────────────────────────────────
# Stratégie : seuil = mean + k*std des scores sur les pièces normales val
val_good_scores = val_scores[val_labels == 0]
threshold_mean  = val_good_scores.mean()
threshold_std   = val_good_scores.std()

# Tester plusieurs valeurs de k
print('=== Calibration du seuil (validation) ===')
print(f'  Scores normaux — mean={threshold_mean:.6f}  std={threshold_std:.6f}')
print()

best_f1, best_k, best_threshold = 0, 0, 0
for k in [1.0, 1.5, 2.0, 2.5, 3.0]:
    threshold = threshold_mean + k * threshold_std
    preds     = (val_scores > threshold).astype(int)
    f1        = f1_score(val_labels, preds, zero_division=0)
    print(f'  k={k:.1f} → seuil={threshold:.6f} | F1={f1:.3f}')
    if f1 > best_f1:
        best_f1, best_k, best_threshold = f1, k, threshold

print(f'\n  ✅ Meilleur seuil : {best_threshold:.6f}  (k={best_k}, F1={best_f1:.3f})')

## 9. Évaluation finale sur le test

In [ ]:
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve)

THRESHOLD = best_threshold
test_preds = (test_scores > THRESHOLD).astype(int)

auc  = roc_auc_score(test_labels, test_scores)
prec = precision_score(test_labels, test_preds, zero_division=0)
rec  = recall_score(test_labels, test_preds, zero_division=0)
f1   = f1_score(test_labels, test_preds, zero_division=0)
cm   = confusion_matrix(test_labels, test_preds)

print('=' * 50)
print('  ÉVALUATION FINALE — TEST — bagel')
print('=' * 50)
print(f'  AUC-ROC    : {auc:.4f}')
print(f'  Précision  : {prec:.4f}')
print(f'  Rappel     : {rec:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  Seuil      : {THRESHOLD:.6f}')
print()
print('  Matrice de confusion :')
print(f'             Prédit Normal  Prédit Défaut')
print(f'  Réel Normal    {cm[0,0]:5d}          {cm[0,1]:5d}')
print(f'  Réel Défaut    {cm[1,0]:5d}          {cm[1,1]:5d}')
print('=' * 50)

In [ ]:
# Visualisation des métriques
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Distribution des scores par classe
good_scores  = test_scores[test_labels == 0]
defect_scores= test_scores[test_labels == 1]
axes[0].hist(good_scores,   bins=20, alpha=0.7, color='#2ecc71', label='Normal',     density=True)
axes[0].hist(defect_scores, bins=20, alpha=0.7, color='#e74c3c', label='Défectueux', density=True)
axes[0].axvline(THRESHOLD, color='black', linestyle='--', linewidth=2, label=f'Seuil={THRESHOLD:.4f}')
axes[0].set_xlabel('Score d\'anomalie (Chamfer Distance)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution des scores')
axes[0].legend(fontsize=8)

# 2. Courbe ROC
fpr, tpr, _ = roc_curve(test_labels, test_scores)
axes[1].plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4)
axes[1].set_xlabel('Taux de faux positifs')
axes[1].set_ylabel('Taux de vrais positifs')
axes[1].set_title('Courbe ROC')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. Scores par type de défaut
defect_types = ['good', 'crack', 'hole', 'contamination', 'combined']
means = []
stds  = []
for dt in defect_types:
    mask = [d == dt for d in test_defects]
    s = test_scores[mask]
    means.append(s.mean() if len(s) > 0 else 0)
    stds.append(s.std()   if len(s) > 0 else 0)

colors = ['#2ecc71'] + ['#e74c3c'] * 4
bars = axes[2].bar(defect_types, means, yerr=stds, capsize=5,
                   color=colors, edgecolor='white', alpha=0.85)
axes[2].axhline(THRESHOLD, color='black', linestyle='--', linewidth=2, label='Seuil')
axes[2].set_xlabel('Type de défaut')
axes[2].set_ylabel('Score moyen (CD)')
axes[2].set_title('Score par type de défaut')
axes[2].tick_params(axis='x', rotation=25)
axes[2].legend()

plt.suptitle(f'Évaluation — Autoencodeur PointNet — bagel  |  AUC={auc:.4f}  F1={f1:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Visualisation — Reconstruction et localisation d'anomalies

In [ ]:
def visualize_reconstruction(model, samples, device, n_show=3, defect_type='crack'):
    """
    Visualise côte à côte :
    - Pièce originale (points colorés par Z)
    - Reconstruction du modèle
    - Carte d'erreur (rouge = zones à haute erreur = anomalies)
    """
    model.eval()

    # Sélectionner les échantillons du type demandé
    target_samples = [s for s in samples if s['defect_type'] == defect_type][:n_show]
    if not target_samples:
        print(f'Aucun échantillon de type {defect_type}')
        return

    fig, axes = plt.subplots(len(target_samples), 3,
                             figsize=(12, 3.5 * len(target_samples)))
    if len(target_samples) == 1:
        axes = [axes]

    for row, sample in enumerate(target_samples):
        # Préparer le point cloud
        pts_raw, _, _ = normalize_pointcloud(load_xyz_tiff(sample['xyz']))
        pts_sampled   = random_sample(pts_raw, N_POINTS)
        pts_tensor    = torch.from_numpy(pts_sampled).float().unsqueeze(0).to(device)

        with torch.no_grad():
            recon, _ = model(pts_tensor)

        pts_np   = pts_sampled                          # (N, 3)
        recon_np = recon.squeeze(0).cpu().numpy()       # (N, 3)

        # Erreur point à point : distance entre chaque point original et
        # son plus proche voisin dans la reconstruction
        diff = pts_np[:, None, :] - recon_np[None, :, :]  # (N, M, 3)
        dist = np.sqrt((diff ** 2).sum(axis=2))             # (N, M)
        error_per_point = dist.min(axis=1)                  # (N,) : erreur locale

        # Normalisation de l'erreur pour la colormap
        err_norm = (error_per_point - error_per_point.min()) / \
                   (error_per_point.max() - error_per_point.min() + 1e-8)

        label  = sample['defect_type']
        color_title = 'red' if label != 'good' else 'green'

        # Original
        axes[row][0].scatter(pts_np[:, 0], pts_np[:, 1],
                             c=pts_np[:, 2], cmap='viridis', s=2, alpha=0.7)
        axes[row][0].set_title(f'Original — {label}', color=color_title, fontsize=9)
        axes[row][0].axis('off')

        # Reconstruction
        axes[row][1].scatter(recon_np[:, 0], recon_np[:, 1],
                             c=recon_np[:, 2], cmap='viridis', s=2, alpha=0.7)
        axes[row][1].set_title('Reconstruction', fontsize=9)
        axes[row][1].axis('off')

        # Carte d'erreur (localisation de l'anomalie)
        sc = axes[row][2].scatter(pts_np[:, 0], pts_np[:, 1],
                                   c=err_norm, cmap='RdYlGn_r', s=3,
                                   vmin=0, vmax=1, alpha=0.8)
        plt.colorbar(sc, ax=axes[row][2], label='Erreur locale')
        axes[row][2].set_title('Carte d\'anomalie\n(rouge = défaut détecté)', fontsize=9)
        axes[row][2].axis('off')

    plt.suptitle(f'Reconstruction & localisation — {defect_type}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'/content/anomaly_map_{defect_type}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Visualisation pour chaque type de défaut
for dtype in ['good', 'crack', 'hole', 'contamination']:
    visualize_reconstruction(model, test_samples, DEVICE, n_show=2, defect_type=dtype)

## 11. Bilan Phase 3 & Décisions pour la Phase 4

In [ ]:
print('=' * 60)
print(f'  BILAN PHASE 3 — Modélisation — {CATEGORY}')
print('=' * 60)
print(f'  Architecture    : Autoencodeur PointNet')
print(f'  Latent dim      : {LATENT_DIM}')
print(f'  Époques         : {N_EPOCHS}')
print(f'  Loss            : Chamfer Distance')
print(f'  Meilleure val   : {best_val_loss:.6f}')
print()
print(f'  ── Métriques test ──')
print(f'  AUC-ROC    : {auc:.4f}  (objectif > 0.85)')
print(f'  Précision  : {prec:.4f}  (objectif > 0.80)')
print(f'  Rappel     : {rec:.4f}  (objectif > 0.80)')
print(f'  F1-Score   : {f1:.4f}  (objectif > 0.80)')
print()
print('  PROCHAINES ÉTAPES (Phase 4 — Évaluation complète) :')
print('  1. Calcul du PRO-score (localisation régionale)')
print('  2. Calcul de l\'IoU sur les masques GT')
print('  3. Analyse par type de défaut (crack vs hole vs contamination)')
print('  4. Comparaison avec baselines publiées MVTec 3D-AD')
print('  5. Analyse des cas d\'erreur (faux positifs / négatifs)')
print('=' * 60)